# Train entity-encoder stack with PairHead trunk + 5 heads (bow+Ebi mix, T=6)

4-layer trainable stack (`PlanetEntityEncoder` → `CrossEntityAttention` → `DualRoleAttention` → `JointRoleAttention`) on top of 3 frozen L0 specialist encoders, capped by a **shared 2-layer trunk + 5 single-Linear heads** producing `pair_logits`, `pair_frac`, `source_act`, `target_aim`, `glob_act` jointly.

## Architecture (head side)

```
                       ctx_now (B, P, 256), source_joint, target_joint
                                  │
                       Project to d_pair=128, broadcast (B, P, P, 6·128)
                                  ▼
                       Linear(768 → 256) → GELU → Linear(256 → 256) → GELU      ← shared trunk
                                  │
                       trunk (B, P, P, 256)
                                  │
        ┌─────────────────┬───────┴───────┬─────────────────────┬───────────────────┐
        ▼                 ▼               ▼                     ▼                   ▼
   pair_head        pair_frac_head   source_act_head      target_aim_head      glob_act_head
   Linear(256→1)    Linear(256→1)    pool→Linear(256→1)   pool→Linear(256→1)   pool→Linear(256→1)
   (B,P,P) BCE      (B,P,P) MSE      (B,P) BCE            (B,P) BCE            (B,) BCE
   pw=600           on sigmoid       pw=100               pw=100               pw=1
```

- `pair_head`: per-cell source→target compatibility, masked by `pair_valid = mask[s] & mask[t] & (s != t)`.
- `pair_frac`: per-cell sigmoid → fraction of source's ships sent to target; masked to positive cells only (where the expert actually launched). Auto-skipped if `pair_ships` is missing from the batch.
- `source_act`: per-planet "this planet launches" (= `pair_labels.any(dim=-1)`), pool-over-targets.
- `target_aim`: per-planet "this planet is targeted" (= `pair_labels.any(dim=-2)`), pool-over-sources.
- `glob_act`: snapshot-level "any action this turn", pool over (P, P).

**Trainable params:** ~3.13M (L1+L2+L3+L4+PairHead). **Frozen L0:** 374k.

## Supervision

- **Teachers (mixed):** `bowwowforeach` (May 10 rank #1, 1800.1) + `Ebi` (rank #3, 1601.5). Combined 555 replays → 60,424 snapshots with **30.8% acted ratio** (the rest are no-op turns, kept so `glob_act` has a positive class).
- **Labels:** the expert launches per turn, walked from raw replay JSON to capture full coalitions. New: `pair_ships (P, P) int32` — total ships sent from each source to each target.
- **Loss:** masked BCE for the binary heads (+ MSE for `pair_frac`).

## Data flow

```
gs://orbit-wars-shipping/entity/
  code.tgz         ~210 KB    shim + agents/transformer_v2 + scripts/build_pair_dataset_orbital_occle.py
  weights.tgz      ~15 MB     frozen planet + fleet + comet d=256 best ckpts
  pair_cache.pt    ~13 GB     bow+Ebi T6 mixed cache (60,424 snapshots, 30.8% acted)
  runs/            outputs land here as <run_name>/{entity_encoder_best.pt, log.json, ...}
```

## Guardrails baked into the cells

1. **Stale extracted code** — `extract` cell wipes `agents/`, `scripts/`, `ckpts/`, `*.pt` (except `pair_cache.pt`) and clears `sys.modules['agents.*']` + `__pycache__`.
2. **Shim asserted** — `verify_code` cell checks the minimal `agents/__init__.py` is loaded.
3. **5-head model present** — assert `m.pair_head.HEAD_NAMES` has all 5 entries and forward returns the 5-key dict.
4. **Cache present + bow+Ebi config** — `stage` cell asserts `config.players == ['bowwowforeach', 'Ebi']`, `acted_min_ratio == 0.3`, T=6.
5. **`pair_ships` available** — assert the cache contains `pair_ships` so `pair_frac` head trains (not skipped).
6. **`d_model` match across L0** — `ckpts` cell asserts `pc==fc==cc==256`.

## 1. Authenticate + pull bundle from GCS

In [ ]:
from google.colab import auth
auth.authenticate_user()
BUCKET = 'gs://orbit-wars-shipping/entity'
print(f'pulling from {BUCKET}')

In [ ]:
import os, subprocess
from pathlib import Path

WORK = Path('/content/orbit-wars')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

# Force-refresh small artifacts; pair_cache.pt is large (~3.8 GB) so
# only re-download if it's missing. Delete it manually if you push a
# fresh cache to GCS and want Colab to re-pull.
for name in ('code.tgz', 'weights.tgz'):
    dst = WORK / name
    if dst.exists():
        dst.unlink()
    subprocess.run(['gsutil', '-q', 'cp', f'{BUCKET}/{name}', str(dst)], check=True)
    print(f'{name}: {dst.stat().st_size/1024/1024:.1f} MB')

pair_pt = WORK / 'pair_cache.pt'
if not pair_pt.exists():
    print('downloading pair_cache.pt (~3.8 GB, ~3 min) ...')
    subprocess.run(['gsutil', 'cp', f'{BUCKET}/pair_cache.pt', str(pair_pt)], check=True)
print(f'pair_cache.pt: {pair_pt.stat().st_size/1024/1024/1024:.2f} GB')

In [ ]:
# Wipe stale extracted code so a previous bundle can't silently shadow
# the new one. The pair_cache.pt is left alone (huge file; the pull
# step's exists() check controls re-download).
!rm -rf agents scripts ckpts data
!find . -maxdepth 1 -name '*.pt' ! -name 'pair_cache.pt' -delete

!tar xzf code.tgz
!tar xzf weights.tgz

import sys
for m in list(sys.modules):
    if m.startswith('agents') or m.startswith('scripts'):
        del sys.modules[m]
import importlib, gc
importlib.invalidate_caches()
gc.collect()
!find . -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null || true

!ls -la

## 1b. Verify the extracted code is the latest one

Fails fast if a stale `agents/__init__.py` slipped through or the `PairHead` is missing (would indicate a pre-pair-head bundle).

In [ ]:
import sys, torch, agents
from pathlib import Path
from agents.transformer_v2.aggregator import (
    CrossEntityAttention, DualRoleAttention, JointRoleAttention, PairHead,
)
from agents.transformer_v2.pretrain.entity_encoder import EntityPretrainModel

print(f'agents module file: {agents.__file__}')
assert 'Minimal' in (agents.__doc__ or ''), \
    'agents/__init__.py is NOT the bundle shim — likely stale extraction; restart kernel.'

# Stack sanity — trunk + 5-head PairHead
m = EntityPretrainModel(d_model=256, n_steps=6)
for attr in ('entity', 'cross', 'dual_role', 'joint_role', 'pair_head'):
    assert hasattr(m, attr), f'EntityPretrainModel missing .{attr} — stale code.tgz.'
assert not hasattr(m, 'source_decoder'), 'stale: source_decoder should be gone.'
assert not hasattr(m, 'target_decoder'), 'stale: target_decoder should be gone.'

# Verify the new 5-head API
expected_heads = {'pair_logits', 'pair_frac', 'source_act', 'target_aim', 'glob_act'}
assert set(m.pair_head.HEAD_NAMES) == expected_heads, \
    f'PairHead.HEAD_NAMES mismatch: {set(m.pair_head.HEAD_NAMES)} vs {expected_heads}'

n_total = sum(p.numel() for p in m.parameters())
n_pair = sum(p.numel() for p in m.pair_head.parameters())
print(f'EntityPretrainModel total trainable: {n_total:,}  (pair_head: {n_pair:,})')
assert n_total > 3_000_000, f'expected ~3.13M params, got {n_total:,} — stale code.tgz.'

# Quick fwd shape check on random input
B, P, F, T, d = 2, 16, 64, 6, 256
planet_tokens = torch.randn(B, T, P, d)
fleet_tokens = torch.randn(B, T, F, d)
routing = {
    'fleet_target_idx': torch.randint(-1, P, (B, T, F)),
    'fleet_source_idx': torch.randint(-1, P, (B, T, F)),
    'fleet_owner_slot': torch.zeros(B, T, F, dtype=torch.long),
    'fleet_ships_log':  torch.zeros(B, T, F),
    'fleet_eta_norm':   torch.ones(B, T, F),
    'fleet_mask':       torch.ones(B, T, F, dtype=torch.bool),
}
planet_mask = torch.ones(B, T, P, dtype=torch.bool)
out = m(planet_tokens, fleet_tokens, routing, planet_mask)
expected_shapes = {
    'pair_logits': (B, P, P), 'pair_frac': (B, P, P),
    'source_act': (B, P),     'target_aim': (B, P),
    'glob_act':   (B,),
}
for name, shape in expected_shapes.items():
    assert out[name].shape == shape, f'{name}: {out[name].shape} != {shape}'
print(f'forward returned 5 heads with correct shapes ✓')
print('all sanity checks passed')

## 2. Stage the pair cache into the layout the pretrain CLI expects

`pretrain/entity_encoder.py`'s default `--pair-cache-path` points at `data/datasets/_pair_cache/bowwowforeach_Ebi_T6/bowwowforeach_Ebi_T6_p64_f1024_all.pt`. We hardlink (or copy) the downloaded `pair_cache.pt` into that location.

In [ ]:
from pathlib import Path
import os

CACHE_DIR = Path('data/datasets/_pair_cache/bowwowforeach_Ebi_T6')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_TARGET = CACHE_DIR / 'bowwowforeach_Ebi_T6_p64_f1024_all.pt'
SOURCE = Path('/content/orbit-wars/pair_cache.pt')

# Plain hardlink — instant on Colab's local FS. Skip the
# try/except symlink fallback; /content/ supports hard links.
if CACHE_TARGET.exists():
    CACHE_TARGET.unlink()
os.link(SOURCE, CACHE_TARGET)

# Size fingerprint — a 13.3 GB file is the bow+Ebi mixed cache;
# the old bow-only (1.4 GB) or OO (3.8 GB) caches would fail here.
size_gb = CACHE_TARGET.stat().st_size / 1024**3
print(f'staged: {CACHE_TARGET}')
print(f'        {size_gb:.2f} GB')
assert size_gb > 10.0, (
    f'pair_cache.pt is only {size_gb:.2f} GB — expected the 13 GB '
    f'bow+Ebi mix. Maybe stale download? Re-pull pair_cache.pt.'
)

# Skip the full CachedPairDataset spot-check here — torch.load on a
# 13 GB pickle takes ~30-60s and the train cell loads it once anyway
# (and will fail-fast on config mismatch via the same path).
print('staging done — config + snapshot validation deferred to train cell')

## 3. Verify torch is importable + GPU available

In [ ]:
import torch
print(f'torch: {torch.__version__}, cuda available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  device: {torch.cuda.get_device_name(0)}')
    print(f'  mem total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 4. Stage all three frozen L0 ckpts into run-dir layout

The pretrain CLI takes `--planet-run-dir` / `--fleet-run-dir` / `--comet-run-dir` (directories), not file paths. We arrange that here and verify each ckpt's `d_model`.

In [ ]:
import shutil
from pathlib import Path
PLANET_RUN_DIR = Path('/content/orbit-wars/ckpts/planet')
FLEET_RUN_DIR  = Path('/content/orbit-wars/ckpts/fleet')
COMET_RUN_DIR  = Path('/content/orbit-wars/ckpts/comet')
for d in (PLANET_RUN_DIR, FLEET_RUN_DIR, COMET_RUN_DIR):
    d.mkdir(parents=True, exist_ok=True)
shutil.copy('/content/orbit-wars/planet_encoder_best.pt', PLANET_RUN_DIR / 'planet_encoder_best.pt')
shutil.copy('/content/orbit-wars/fleet_encoder_best.pt',  FLEET_RUN_DIR  / 'fleet_encoder_best.pt')
shutil.copy('/content/orbit-wars/comet_past_best.pt',     COMET_RUN_DIR  / 'comet_past_best.pt')

import torch
pc = torch.load(PLANET_RUN_DIR / 'planet_encoder_best.pt', map_location='cpu', weights_only=False)
fc = torch.load(FLEET_RUN_DIR  / 'fleet_encoder_best.pt',  map_location='cpu', weights_only=False)
cc = torch.load(COMET_RUN_DIR  / 'comet_past_best.pt',     map_location='cpu', weights_only=False)
print(f'planet ckpt: d_model={pc["config"]["d_model"]}, epoch={pc["epoch"]}, use_traj_branch={pc["config"].get("use_traj_branch")}')
print(f'fleet  ckpt: d_model={fc["config"]["d_model"]}, epoch={fc["epoch"]}')
print(f'comet  ckpt: d_model={cc["config"]["d_model"]}, epoch={cc["epoch"]}, input_dim={cc["config"].get("input_dim")}')

assert pc['config']['d_model'] == fc['config']['d_model'] == cc['config']['d_model'] == 256, \
    'all three L0 encoders must be d=256'

## 5. Train

The entity-encoder CLI will:

1. Read upstream `d_model` from each ckpt config and size the L0 encoders accordingly; freeze them.
2. Load `CachedPairDataset` from `--pair-cache-path` and split episodes 80/10/10 (`--val-frac 0.10 --test-frac 0.10`).
3. Per batch: stack T=6 history lazily, run L0 frozen, route via `where(is_comet, comet, planet)`, run L1 → L2 → L3 → L4 → PairHead.
4. Per epoch: `pair_logits` BCE-with-logits masked by `pair_valid`, with `pos_weight=600` on the train loss (val/test unweighted).
5. Print per-epoch table with `recall_true`, `recall_false`, `recall_at_{1, 5, 10}`.

In [ ]:
D_MODEL              = 256
BATCH_SIZE           = 128      # large batch; halve to 64 if you OOM on T4
EPOCHS               = 30
LR                   = 5e-5     # smaller LR for the 5-head jointly-trained stack
WEIGHT_DECAY         = 1e-4
SEED                 = 1729
MAX_PLANETS          = 64
MAX_FLEETS           = 1024
PAIR_POS_WEIGHT      = 600.0    # pair-cell BCE imbalance
SOURCE_ACT_POS_WEIGHT = 100.0   # per-source BCE imbalance
TARGET_AIM_POS_WEIGHT = 100.0   # per-target BCE imbalance
GLOB_ACT_POS_WEIGHT   = 1.0     # snapshot-level (already ~30% positive)
VAL_FRAC             = 0.10
TEST_FRAC            = 0.10
DEVICE               = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'

import time
TS = time.strftime('%Y%m%d-%H%M%S')
OUT_DIR = f'data/runs/entity/bowEbi_pair5head_d{D_MODEL}_lr{LR:g}_b{BATCH_SIZE}_{EPOCHS}ep_{TS}'
print('out dir:', OUT_DIR)

In [ ]:
!python -u -m agents.transformer_v2.pretrain.entity_encoder \
  --planet-run-dir $PLANET_RUN_DIR \
  --fleet-run-dir  $FLEET_RUN_DIR \
  --comet-run-dir  $COMET_RUN_DIR \
  --pair-cache-path data/datasets/_pair_cache/bowwowforeach_Ebi_T6/bowwowforeach_Ebi_T6_p64_f1024_all.pt \
  --out-dir $OUT_DIR \
  --d-model $D_MODEL \
  --batch-size $BATCH_SIZE \
  --epochs $EPOCHS \
  --lr $LR \
  --weight-decay $WEIGHT_DECAY \
  --max-planets $MAX_PLANETS \
  --max-fleets $MAX_FLEETS \
  --pair-pos-weight $PAIR_POS_WEIGHT \
  --source-act-pos-weight $SOURCE_ACT_POS_WEIGHT \
  --target-aim-pos-weight $TARGET_AIM_POS_WEIGHT \
  --glob-act-pos-weight $GLOB_ACT_POS_WEIGHT \
  --val-frac $VAL_FRAC \
  --test-frac $TEST_FRAC \
  --seed $SEED \
  --device $DEVICE

## 6. Push the trained run back to GCS

In [ ]:
import subprocess
from pathlib import Path
src = Path(OUT_DIR)
assert src.is_dir(), src
# Pass src without trailing slash + destination parent so gsutil cp -r
# creates runs/<src.name>/<files>... without doubling the dir name.
dst_parent = f'{BUCKET}/runs/'
subprocess.run(['gsutil', '-m', 'cp', '-r', str(src), dst_parent], check=True)
dst = f'{dst_parent}{src.name}/'
print(f'uploaded to: {dst}')
subprocess.run(['gsutil', 'ls', '-lh', dst], check=False)